# PS S6E4 - exp_20260410_031_hc_top_candidates

目的:
 - 029 までで残った上位 stack 候補に対して Hill Climbing を行う
 - 対象は 018+025+026 と 018+024+026 の2セット
 - 025 単体を強い基準点として比較する

方針:
 - まず biased proba を入力素材として使う
 - HC は rank average ではなく probability weighted average で行う
 - CV を見て best weights を出す
 - 031 では Nelder-Mead はやらない
 - 031 の結果が良ければ次に 032 で Nelder-Mead

## Imports

In [1]:
import os
import json
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import balanced_accuracy_score

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 300)
pd.set_option("display.width", 200)

## CFG

In [2]:
class CFG:
    EXP_ID = "exp_20260410_031_hc_top_candidates"
    EXP_NAME = "hc_top_candidates"
    SAVE_DIR = Path(f"/kaggle/working/{EXP_ID}")
    SAVE_DIR.mkdir(parents=True, exist_ok=True)

    TRAIN_PATH = "/kaggle/input/competitions/playground-series-s6e4/train.csv"
    TEST_PATH = "/kaggle/input/competitions/playground-series-s6e4/test.csv"
    SAMPLE_SUB_PATH = "/kaggle/input/competitions/playground-series-s6e4/sample_submission.csv"
    NPY_PATH = "/kaggle/input/datasets/mizushimatoshihiko/npy-files/"

    TARGET_COL = "Irrigation_Need"
    ID_COL = "id"
    SEED = 42
    N_CLASSES = 3

    HC_STEP_GRID = [
        0.20,
        0.10,
        0.05,
        0.02,
        0.01,
        0.005,
    ]

    MAX_ROUNDS_PER_STEP = 50

## utility

In [3]:
def seed_everything(seed):
    np.random.seed(seed)
    random.seed(seed)

def normalize_proba(p):
    p = np.asarray(p, dtype=float)
    p = np.clip(p, 1e-15, None)
    row_sum = p.sum(axis=1, keepdims=True)
    row_sum = np.where(row_sum == 0, 1.0, row_sum)
    return p / row_sum

def balanced_acc_score_mc(y_true, proba_or_pred):
    arr = np.asarray(proba_or_pred)
    if arr.ndim == 2:
        pred = np.argmax(arr, axis=1)
    else:
        pred = arr
    return float(balanced_accuracy_score(y_true, pred))

def weighted_average_proba(proba_list, weights):
    weights = np.asarray(weights, dtype=float)
    weights = np.clip(weights, 0.0, None)
    if weights.sum() == 0:
        weights = np.ones_like(weights)
    weights = weights / weights.sum()

    out = np.zeros_like(proba_list[0], dtype=float)
    for w, p in zip(weights, proba_list):
        out += w * p
    return normalize_proba(out)

seed_everything(CFG.SEED)

## load target / submission template

In [4]:
train_df = pd.read_csv(CFG.TRAIN_PATH)
test_df = pd.read_csv(CFG.TEST_PATH)
sample_sub = pd.read_csv(CFG.SAMPLE_SUB_PATH)

target2idx = {v: i for i, v in enumerate(train_df[CFG.TARGET_COL].unique())}
idx2target = {v: k for k, v in target2idx.items()}
y = train_df[CFG.TARGET_COL].map(target2idx).values.astype(int)

print(train_df.shape, test_df.shape)
print(target2idx)

(630000, 21) (270000, 20)
{'Low': 0, 'Medium': 1, 'High': 2}


## load npy

In [5]:
model_dict = {
    "018": (
        np.load(CFG.NPY_PATH + "oof_cat_pairwise_te_bias_from_shared_min_proba_biased.npy"),
        np.load(CFG.NPY_PATH + "pred_cat_pairwise_te_bias_from_shared_min_proba_biased.npy"),
    ),
    "024": (
        np.load(CFG.NPY_PATH + "oof_xgb_digit_orderedte_min_proba_biased.npy"),
        np.load(CFG.NPY_PATH + "pred_xgb_digit_orderedte_min_proba_biased.npy"),
    ),
    "025": (
        np.load(CFG.NPY_PATH + "oof_lgb_digit_te_min_proba_biased.npy"),
        np.load(CFG.NPY_PATH + "pred_lgb_digit_te_min_proba_biased.npy"),
    ),
    "026": (
        np.load(CFG.NPY_PATH + "oof_realmlp_pair_te_min_proba_biased.npy"),
        np.load(CFG.NPY_PATH + "pred_realmlp_pair_te_min_proba_biased.npy"),
    ),
}

## oof, pred shapes

In [6]:
for name, (oof_arr, pred_arr) in model_dict.items():
    print(name)
    print("  oof shape:", oof_arr.shape, " row sum head:", np.round(oof_arr[:5].sum(axis=1), 6))
    print("  pred shape:", pred_arr.shape, " row sum head:", np.round(pred_arr[:5].sum(axis=1), 6))

018
  oof shape: (630000, 3)  row sum head: [1. 1. 1. 1. 1.]
  pred shape: (270000, 3)  row sum head: [1. 1. 1. 1. 1.]
024
  oof shape: (630000, 3)  row sum head: [1. 1. 1. 1. 1.]
  pred shape: (270000, 3)  row sum head: [1. 1. 1. 1. 1.]
025
  oof shape: (630000, 3)  row sum head: [1. 1. 1. 1. 1.]
  pred shape: (270000, 3)  row sum head: [1. 1. 1. 1. 1.]
026
  oof shape: (630000, 3)  row sum head: [1. 1. 1. 1. 1.]
  pred shape: (270000, 3)  row sum head: [1. 1. 1. 1. 1.]


## candidate sets

In [7]:
candidate_sets = {
    "baseline_025": ["025"],
    "hc_main_018_025_026": ["018", "025", "026"],
    "hc_compare_018_024_026": ["018", "024", "026"],
}

candidate_sets

{'baseline_025': ['025'],
 'hc_main_018_025_026': ['018', '025', '026'],
 'hc_compare_018_024_026': ['018', '024', '026']}

## baseline result

In [8]:
baseline_oof = model_dict["025"][0].copy()
baseline_pred = model_dict["025"][1].copy()
baseline_cv = balanced_acc_score_mc(y, baseline_oof)

baseline_result = {
    "set_name": "baseline_025",
    "models": ["025"],
    "weights": [1.0],
    "raw_cv_ba": baseline_cv,
}

baseline_result

{'set_name': 'baseline_025',
 'models': ['025'],
 'weights': [1.0],
 'raw_cv_ba': 0.9793964215510967}

## HC helper

In [9]:
def hill_climb_weights(proba_list, y_true, step_grid, max_rounds_per_step=50):
    n_models = len(proba_list)

    # 初期値: 等重み
    current_weights = np.ones(n_models, dtype=float) / n_models
    current_oof = weighted_average_proba(proba_list, current_weights)
    current_score = balanced_acc_score_mc(y_true, current_oof)

    history = []
    history.append({
        "stage": "init",
        "weights": current_weights.tolist(),
        "cv": current_score,
    })

    for step in step_grid:
        improved_any = False

        for _ in range(max_rounds_per_step):
            improved = False
            best_local_score = current_score
            best_local_weights = current_weights.copy()

            for i in range(n_models):
                for direction in [-1.0, 1.0]:
                    trial_weights = current_weights.copy()
                    trial_weights[i] += direction * step

                    if np.any(trial_weights < 0):
                        continue

                    trial_oof = weighted_average_proba(proba_list, trial_weights)
                    trial_score = balanced_acc_score_mc(y_true, trial_oof)

                    if trial_score > best_local_score + 1e-12:
                        best_local_score = trial_score
                        best_local_weights = trial_weights.copy()
                        improved = True

            if improved:
                current_weights = best_local_weights.copy()
                current_oof = weighted_average_proba(proba_list, current_weights)
                current_score = best_local_score
                improved_any = True

                history.append({
                    "stage": f"step_{step}",
                    "weights": current_weights.tolist(),
                    "cv": current_score,
                })
            else:
                break

        if not improved_any:
            history.append({
                "stage": f"step_{step}_no_improve",
                "weights": current_weights.tolist(),
                "cv": current_score,
            })

    current_weights = np.clip(current_weights, 0.0, None)
    current_weights = current_weights / current_weights.sum()

    final_oof = weighted_average_proba(proba_list, current_weights)
    final_score = balanced_acc_score_mc(y_true, final_oof)

    return current_weights, final_oof, final_score, history

## run HC

In [10]:
hc_results = []
hc_outputs = {}

for set_name, model_names in candidate_sets.items():
    print("=" * 100)
    print(set_name, model_names)
    print("=" * 100)

    if len(model_names) == 1:
        oof_blend = model_dict[model_names[0]][0].copy()
        pred_blend = model_dict[model_names[0]][1].copy()
        weights = np.array([1.0], dtype=float)
        cv_score = balanced_acc_score_mc(y, oof_blend)
        history = [{"stage": "baseline", "weights": [1.0], "cv": cv_score}]
    else:
        proba_list_oof = [model_dict[m][0] for m in model_names]
        proba_list_pred = [model_dict[m][1] for m in model_names]

        weights, oof_blend, cv_score, history = hill_climb_weights(
            proba_list=proba_list_oof,
            y_true=y,
            step_grid=CFG.HC_STEP_GRID,
            max_rounds_per_step=CFG.MAX_ROUNDS_PER_STEP,
        )

        pred_blend = weighted_average_proba(proba_list_pred, weights)

    hc_results.append({
        "set_name": set_name,
        "models": "+".join(model_names),
        "weights": weights.tolist(),
        "raw_cv_ba": cv_score,
    })

    hc_outputs[set_name] = {
        "models": model_names,
        "weights": weights.tolist(),
        "oof": oof_blend,
        "pred": pred_blend,
        "history": history,
    }

    print("weights =", weights.tolist())
    print("raw_cv_ba =", cv_score)

baseline_025 ['025']
weights = [1.0]
raw_cv_ba = 0.9793964215510967
hc_main_018_025_026 ['018', '025', '026']
weights = [0.273224043715847, 0.5191256830601094, 0.20765027322404372]
raw_cv_ba = 0.9800547965657039
hc_compare_018_024_026 ['018', '024', '026']
weights = [0.11845730027548208, 0.5234159779614325, 0.3581267217630854]
raw_cv_ba = 0.9800100572223786


## results table

In [11]:
hc_results_df = pd.DataFrame(hc_results).sort_values("raw_cv_ba", ascending=False).reset_index(drop=True)
display(hc_results_df)

,set_name,models,weights,raw_cv_ba
0,hc_main_018_025_026,018+025+026,"[0.273224043715847, 0.5191256830601094, 0.2076...",0.980055
1,hc_compare_018_024_026,018+024+026,"[0.11845730027548208, 0.5234159779614325, 0.35...",0.980010
2,baseline_025,025,[1.0],0.979396


## save results

In [12]:
hc_results_df.to_csv(CFG.SAVE_DIR / "hc_results.csv", index=False)

with open(CFG.SAVE_DIR / "hc_results.json", "w", encoding="utf-8") as f:
    json.dump(hc_results, f, ensure_ascii=False, indent=2)

with open(CFG.SAVE_DIR / "hc_histories.json", "w", encoding="utf-8") as f:
    json.dump({k: v["history"] for k, v in hc_outputs.items()}, f, ensure_ascii=False, indent=2)

print("saved to:", CFG.SAVE_DIR)

saved to: /kaggle/working/exp_20260410_031_hc_top_candidates


## save best HC submission

In [13]:
best_row = hc_results_df.iloc[0]
best_set = best_row["set_name"]
best_obj = hc_outputs[best_set]

best_pred_idx = np.argmax(best_obj["pred"], axis=1)

submission = sample_sub.copy()
submission[CFG.TARGET_COL] = pd.Series(best_pred_idx).map(idx2target)
submission.to_csv(CFG.SAVE_DIR / f"submission_{best_set}_hc.csv", index=False)

np.save(CFG.SAVE_DIR / f"oof_{best_set}_hc_proba.npy", best_obj["oof"])
np.save(CFG.SAVE_DIR / f"pred_{best_set}_hc_proba.npy", best_obj["pred"])

print("best_set =", best_set)
print("best_models =", best_obj["models"])
print("best_weights =", best_obj["weights"])
display(submission.head())

best_set = hc_main_018_025_026
best_models = ['018', '025', '026']
best_weights = [0.273224043715847, 0.5191256830601094, 0.20765027322404372]


,id,Irrigation_Need
0,630000,Low
1,630001,Low
2,630002,Low
3,630003,Low
4,630004,Low


## save meta note

In [14]:
meta_note = {
    "candidate_sets": candidate_sets,
    "hc_step_grid": CFG.HC_STEP_GRID,
    "max_rounds_per_step": CFG.MAX_ROUNDS_PER_STEP,
    "best_set": best_set,
    "best_models": best_obj["models"],
    "best_weights": best_obj["weights"],
}

with open(CFG.SAVE_DIR / "meta_note.json", "w", encoding="utf-8") as f:
    json.dump(meta_note, f, ensure_ascii=False, indent=2)

print("done")

done
